# TP 2 — Exercice 4 : Morphologie mathématique

Image : `tp2_exo4_globules_rouges_cells.jpg` (`Globules_rouges.jpg`).
1. Binarisation dans `imb1` et application de l'**érosion**, **dilatation**, **ouverture** et **fermeture** ($3\times 3$, $5\times 5$, $7\times 7$) via OpenCV puis fonctions personnelles.
2. Séparation et comptage des globules rouges par combinaison morphologique + transformée de distance / Watershed.
3. Squelettisation morphologique par l'algorithme de **Zhang-Suen** (`cv2.ximgproc.thinning` + implémentation NumPy autonome).

In [ ]:
from pathlib import Path
import math
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "tp1_tp2_images").is_dir():
            return candidate
    raise FileNotFoundError(
        "Impossible de trouver le dossier 'tp1_tp2_images'. "
        "Lancez Jupyter depuis la racine TAI ou depuis tp1_labs / tp2_labs."
    )

ROOT_DIR = find_project_root()
IMG_DIR = ROOT_DIR / "tp1_tp2_images"
OUT_DIR = ROOT_DIR / "tp2_labs" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_image(filename, flags=cv2.IMREAD_COLOR):
    path = IMG_DIR / filename
    img = cv2.imread(str(path), flags)
    if img is None:
        raise FileNotFoundError(f"Image introuvable ou illisible: {path}")
    return img

def show_images(items, cols=3, figsize_per_cell=(5, 4)):
    n = len(items)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_cell[0], rows * figsize_per_cell[1]))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n:]:
        ax.axis("off")
    for ax, (title, img) in zip(axes, items):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        elif img.shape[2] == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        elif img.shape[2] == 4:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA))
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"IMG_DIR  : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"OUT_DIR  : {OUT_DIR}")

In [ ]:
# Lecture en niveaux de gris et binarisation automatique (globules en blanc = 255, fond en noir = 0)
im1 = load_image("tp2_exo4_globules_rouges_cells.jpg", cv2.IMREAD_GRAYSCALE)
otsu_t, _ = cv2.threshold(im1, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
border_vals = np.concatenate([im1[0, :], im1[-1, :], im1[:, 0], im1[:, -1]])
if np.median(border_vals) > otsu_t:
    _, imb1 = cv2.threshold(im1, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
else:
    _, imb1 = cv2.threshold(im1, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f"Seuil d'Otsu retenu : {otsu_t:.1f} | Pixels foreground : {np.mean(imb1 > 0)*100:.1f}%")

# Fonctions morphologiques personnelles
def ma_erosion(img, kernel):
    kh, kw = kernel.shape
    rh, rw = kh // 2, kw // 2
    h, w = img.shape
    padded = np.pad(img, ((rh, rh), (rw, rw)), mode="edge")
    out = np.full((h, w), 255, dtype=img.dtype)
    for dy in range(kh):
        for dx in range(kw):
            if kernel[dy, dx] > 0:
                out = np.minimum(out, padded[dy:dy + h, dx:dx + w])
    return out

def ma_dilatation(img, kernel):
    kh, kw = kernel.shape
    rh, rw = kh // 2, kw // 2
    h, w = img.shape
    padded = np.pad(img, ((rh, rh), (rw, rw)), mode="edge")
    out = np.zeros((h, w), dtype=img.dtype)
    for dy in range(kh):
        for dx in range(kw):
            if kernel[dy, dx] > 0:
                out = np.maximum(out, padded[dy:dy + h, dx:dx + w])
    return out

def ma_ouverture(img, kernel):
    return ma_dilatation(ma_erosion(img, kernel), kernel)

def ma_fermeture(img, kernel):
    return ma_erosion(ma_dilatation(img, kernel), kernel)

# 1) Test 3x3, 5x5, 7x7 OpenCV vs Perso
viz_items = [("Originale (NG)", im1), ("Binaire Otsu (imb1)", imb1)]
for ksize in (3, 5, 7):
    es = np.ones((ksize, ksize), dtype=np.uint8)
    cv_ero = cv2.erode(imb1, es)
    cv_dil = cv2.dilate(imb1, es)
    cv_ouv = cv2.morphologyEx(imb1, cv2.MORPH_OPEN, es)
    cv_fer = cv2.morphologyEx(imb1, cv2.MORPH_CLOSE, es)

    my_ero = ma_erosion(imb1, es)
    my_dil = ma_dilatation(imb1, es)
    my_ouv = ma_ouverture(imb1, es)
    my_fer = ma_fermeture(imb1, es)

    print(
        f"ES {ksize}x{ksize} -> Égalité stricte OpenCV/Perso : "
        f"Ero={np.array_equal(cv_ero, my_ero)}, "
        f"Dil={np.array_equal(cv_dil, my_dil)}, "
        f"Ouv={np.array_equal(cv_ouv, my_ouv)}, "
        f"Fer={np.array_equal(cv_fer, my_fer)}"
    )
    if ksize == 5:
        viz_items.extend([
            ("Érosion 5x5", my_ero),
            ("Dilatation 5x5", my_dil),
            ("Ouverture 5x5", my_ouv),
            ("Fermeture 5x5", my_fer),
        ])

show_images(viz_items, cols=3)

In [ ]:
# 2) Séparation et comptage des globules rouges
# Étape A : Fermeture puis remplissage des trous internes + ouverture pour supprimer le bruit
k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
closed = cv2.morphologyEx(imb1, cv2.MORPH_CLOSE, k_small, iterations=2)

# Remplissage de trous via contours externes
contours_fill, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
filled = np.zeros_like(closed)
cv2.drawContours(filled, contours_fill, -1, 255, thickness=cv2.FILLED)
clean = cv2.morphologyEx(filled, cv2.MORPH_OPEN, k_small, iterations=1)

raw_num, _ = cv2.connectedComponents(clean)

# Étape B : Transformée en distance + marqueurs + Watershed pour séparer les globules accolés
dist = cv2.distanceTransform(clean, cv2.DIST_L2, 5)
dist_smooth = cv2.GaussianBlur(dist, (5, 5), 0)
sure_bg = cv2.dilate(clean, k_small, iterations=2)
_, sure_fg = cv2.threshold(dist_smooth, 0.42 * dist_smooth.max(), 255, cv2.THRESH_BINARY)
sure_fg = sure_fg.astype(np.uint8)
unknown = cv2.subtract(sure_bg, sure_fg)

n_markers, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
markers[unknown == 255] = 0
img_color = cv2.cvtColor(im1, cv2.COLOR_GRAY2BGR)
markers_ws = cv2.watershed(img_color.copy(), markers)

# Comptage final et annotation
cell_count = 0
annotated = img_color.copy()
annotated[markers_ws == -1] = [0, 0, 255]  # frontières watershed en rouge
for label_id in range(2, n_markers + 1):
    mask_cell = (markers_ws == label_id).astype(np.uint8)
    area = int(np.sum(mask_cell))
    if area < 40:
        continue
    cell_count += 1
    ys, xs = np.where(mask_cell > 0)
    cy, cx = int(np.mean(ys)), int(np.mean(xs))
    cv2.putText(annotated, str(cell_count), (cx - 8, cy + 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 0), 1, cv2.LINE_AA)

print(f"Composantes avant séparation : {raw_num - 1}")
print(f"Nombre de globules rouges comptés après séparation morphologique : {cell_count}")

show_images([
    ("Masque nettoyé", clean),
    ("Carte de distance", cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)),
    ("Marqueurs séparés (sure_fg)", sure_fg),
    (f"Globules séparés & comptés ({cell_count})", annotated),
], cols=4)

In [ ]:
# 3) Squelettisation de Zhang-Suen
def zhang_suen_thinning_perso(binary_img):
    img = (binary_img > 0).astype(np.uint8)
    changed = True
    while changed:
        changed = False
        for step in (0, 1):
            p = np.pad(img, 1, mode="constant", constant_values=0)
            p2, p3, p4 = p[:-2, 1:-1], p[:-2, 2:], p[1:-1, 2:]
            p5, p6, p7 = p[2:, 2:], p[2:, 1:-1], p[2:, :-2]
            p8, p9 = p[1:-1, :-2], p[:-2, :-2]
            b = p2 + p3 + p4 + p5 + p6 + p7 + p8 + p9
            neighbors = [p2, p3, p4, p5, p6, p7, p8, p9, p2]
            a = np.zeros_like(img, dtype=np.uint8)
            for k in range(8):
                a += ((neighbors[k] == 0) & (neighbors[k + 1] == 1)).astype(np.uint8)
            if step == 0:
                c1 = (p2 * p4 * p6) == 0
                c2 = (p4 * p6 * p8) == 0
            else:
                c1 = (p2 * p4 * p8) == 0
                c2 = (p2 * p6 * p8) == 0
            cond = (img == 1) & (b >= 2) & (b <= 6) & (a == 1) & c1 & c2
            if np.any(cond):
                img[cond] = 0
                changed = True
    return (img * 255).astype(np.uint8)

skel_perso = zhang_suen_thinning_perso(clean)
items_skel = [("Masque binaire nettoyé", clean), ("Squelette Zhang-Suen (Perso)", skel_perso)]

if hasattr(cv2, "ximgproc"):
    skel_cv = cv2.ximgproc.thinning(clean, thinningType=cv2.ximgproc.THINNING_ZHANGSUEN)
    print("cv2.ximgproc.thinning disponible ! Égalité avec version perso :", np.array_equal(skel_cv, skel_perso))
    items_skel.append(("Squelette OpenCV ximgproc", skel_cv))
else:
    print("Module cv2.ximgproc absent (opencv-contrib-python non installé) -> fallback perso utilisé.")

show_images(items_skel, cols=len(items_skel))